In [1]:
import os
import time
import datetime
import json
import re
import pandas as pd
from dotenv import load_dotenv
from datasets import load_dataset
import openai
import anthropic
from google import genai
from google.genai import types
import mlflow

load_dotenv()

if "GOOGLE_API_KEY" in os.environ:
    os.environ.pop("GOOGLE_API_KEY")

openai_client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
anthropic_client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
gemini_client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

print("Loading CoNLL-2003 dataset...")
dataset = load_dataset("eriktks/conll2003", split="test", revision="refs/convert/parquet")
samples = dataset.select(range(100))

label_map = {
    0: "O", 1: "B-PER", 2: "I-PER",
    3: "B-ORG", 4: "I-ORG",
    5: "B-LOC", 6: "I-LOC",
    7: "B-MISC", 8: "I-MISC"
}

# FIX 2: Added MISC to prompt — matches ground truth labels
SYSTEM_PROMPT = (
    "You are a Named Entity Recognition system. "
    "Extract all named entities from the text. "
    "Return ONLY a JSON object with keys: PER (persons), ORG (organizations), LOC (locations), MISC (miscellaneous). "
    'Example: {"PER": ["John Smith"], "ORG": ["Apple"], "LOC": ["California"], "MISC": ["FIFA"]}'
)

def tokens_to_text(tokens):
    return " ".join(tokens)

def extract_entities_openai(text):
    start = time.time()
    response = openai_client.chat.completions.create(
        model="gpt-4o",  # FIX 1: thesis spec
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Text: {text}"}
        ],
        max_tokens=150,
        temperature=0.0
    )
    latency = time.time() - start
    return response.choices[0].message.content.strip(), latency

def extract_entities_anthropic(text):
    start = time.time()
    response = anthropic_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=150,
        temperature=0.0,  # FIX: match GPT-4o/Gemini's greedy decoding for a fair comparison
        system=SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"Text: {text}"}]
    )
    latency = time.time() - start
    return response.content[0].text.strip(), latency

def extract_entities_gemini(text):
    start = time.time()
    try:
        response = gemini_client.models.generate_content(
            model="gemini-2.5-flash",
            contents=f"{SYSTEM_PROMPT}\n\nText: {text}",
            config=types.GenerateContentConfig(
                max_output_tokens=150,
                temperature=0.0,
                thinking_config=types.ThinkingConfig(thinking_budget=0)
            )
        )
        latency = time.time() - start
        return response.text.strip() if response.text else "{}", latency
    except Exception as e:
        latency = time.time() - start
        print(f"Gemini error: {e}")
        return "{}", latency

def get_true_entities(sample):
    """Convert BIO tags to entity dict including MISC."""
    entities = {"PER": [], "ORG": [], "LOC": [], "MISC": []}
    tokens = sample["tokens"]
    ner_tags = sample["ner_tags"]
    current_entity = []
    current_type = None

    for token, tag in zip(tokens, ner_tags):
        label = label_map[tag]
        if label.startswith("B-"):
            if current_entity and current_type:
                entities[current_type].append(" ".join(current_entity))
            current_entity = [token]
            current_type = label[2:]
        elif label.startswith("I-") and current_entity:
            current_entity.append(token)
        else:
            if current_entity and current_type:
                entities[current_type].append(" ".join(current_entity))
            current_entity = []
            current_type = None

    if current_entity and current_type:
        entities[current_type].append(" ".join(current_entity))

    return {k: v for k, v in entities.items() if v}

def parse_entity_response(response_text):
    """Parse JSON response into flat lowercased set."""
    try:
        clean = re.sub(r"```json|```", "", response_text).strip()
        parsed = json.loads(clean)
        entities = []
        for key in ["PER", "ORG", "LOC", "MISC"]:  # FIX 2: include MISC
            entities.extend([e.lower().strip() for e in parsed.get(key, [])])
        return set(entities)
    except Exception:
        return set()

def compute_f1(true_entities_dict, predicted_set):
    """Exact match F1 at entity level."""
    true_flat = set(
        e.lower().strip()
        for entities in true_entities_dict.values()
        for e in entities
    )
    if not true_flat and not predicted_set:
        return 1.0, 1.0, 1.0

    tp = len(true_flat & predicted_set)
    fp = len(predicted_set - true_flat)
    fn = len(true_flat - predicted_set)

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1        = (2 * precision * recall / (precision + recall)
                 if (precision + recall) > 0 else 0.0)
    return precision, recall, f1

# --- Evaluation ---
mlflow.set_experiment("conll2003_baseline")
results = []

with mlflow.start_run(run_name="ner_baseline_v2_fixed"):
    mlflow.log_param("sample_count", 100)
    mlflow.log_param("openai_model", "gpt-4o")
    mlflow.log_param("anthropic_model", "claude-haiku-4-5-20251001")
    mlflow.log_param("gemini_model", "gemini-2.5-flash")
    mlflow.log_param("misc_fix", "included")

    oai_precisions, oai_recalls, oai_f1s = [], [], []
    ant_precisions, ant_recalls, ant_f1s = [], [], []
    gem_precisions, gem_recalls, gem_f1s = [], [], []
    oai_lats, ant_lats, gem_lats = [], [], []

    print("Evaluating...")
    for i, sample in enumerate(samples):
        text = tokens_to_text(sample["tokens"])
        true_entities = get_true_entities(sample)

        oai_response, oai_lat = extract_entities_openai(text)
        ant_response, ant_lat = extract_entities_anthropic(text)
        gem_response, gem_lat = extract_entities_gemini(text)

        oai_pred = parse_entity_response(oai_response)
        ant_pred = parse_entity_response(ant_response)
        gem_pred = parse_entity_response(gem_response)

        oai_p, oai_r, oai_f1 = compute_f1(true_entities, oai_pred)
        ant_p, ant_r, ant_f1 = compute_f1(true_entities, ant_pred)
        gem_p, gem_r, gem_f1 = compute_f1(true_entities, gem_pred)

        oai_precisions.append(oai_p); oai_recalls.append(oai_r); oai_f1s.append(oai_f1)
        ant_precisions.append(ant_p); ant_recalls.append(ant_r); ant_f1s.append(ant_f1)
        gem_precisions.append(gem_p); gem_recalls.append(gem_r); gem_f1s.append(gem_f1)
        oai_lats.append(oai_lat); ant_lats.append(ant_lat); gem_lats.append(gem_lat)

        results.append({
            "text": text[:80],
            "true_entities": str(true_entities),
            "openai_response": oai_response,
            "anthropic_response": ant_response,
            "gemini_response": gem_response,
            "openai_correct": int(oai_f1 >= 0.5),
            "anthropic_correct": int(ant_f1 >= 0.5),
            "gemini_correct": int(gem_f1 >= 0.5),
            "openai_f1": round(oai_f1, 4),
            "anthropic_f1": round(ant_f1, 4),
            "gemini_f1": round(gem_f1, 4),
            "openai_latency_s": round(oai_lat, 3),
            "anthropic_latency_s": round(ant_lat, 3),
            "gemini_latency_s": round(gem_lat, 3),
        })

        if i % 10 == 0:
            print(f"Progress: {i}/100")

    # Aggregate
    avg = lambda lst: sum(lst) / len(lst)

    mlflow.log_metric("openai_precision",       avg(oai_precisions))
    mlflow.log_metric("openai_recall",          avg(oai_recalls))
    mlflow.log_metric("openai_f1",              avg(oai_f1s))
    mlflow.log_metric("anthropic_precision",    avg(ant_precisions))
    mlflow.log_metric("anthropic_recall",       avg(ant_recalls))
    mlflow.log_metric("anthropic_f1",           avg(ant_f1s))
    mlflow.log_metric("gemini_precision",       avg(gem_precisions))
    mlflow.log_metric("gemini_recall",          avg(gem_recalls))
    mlflow.log_metric("gemini_f1",              avg(gem_f1s))
    mlflow.log_metric("openai_avg_latency_s",   avg(oai_lats))
    mlflow.log_metric("anthropic_avg_latency_s",avg(ant_lats))
    mlflow.log_metric("gemini_avg_latency_s",   avg(gem_lats))

    print(f"\n{'='*55}")
    print(f"{'Metric':<22} {'GPT-4o':>8} {'Haiku':>10} {'Gemini':>10}")
    print(f"{'='*55}")
    print(f"{'Precision':<22} {avg(oai_precisions):>8.3f} {avg(ant_precisions):>10.3f} {avg(gem_precisions):>10.3f}")
    print(f"{'Recall':<22} {avg(oai_recalls):>8.3f} {avg(ant_recalls):>10.3f} {avg(gem_recalls):>10.3f}")
    print(f"{'F1 Score':<22} {avg(oai_f1s):>8.3f} {avg(ant_f1s):>10.3f} {avg(gem_f1s):>10.3f}")
    print(f"{'Avg Latency (s)':<22} {avg(oai_lats):>8.3f} {avg(ant_lats):>10.3f} {avg(gem_lats):>10.3f}")
    print(f"{'='*55}")

os.makedirs("../logs", exist_ok=True)
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
csv_path = f"../logs/conll_baseline_{timestamp}.csv"
pd.DataFrame(results).to_csv(csv_path, index=False)
print(f"Saved to {csv_path}")

Loading CoNLL-2003 dataset...
Evaluating...
Progress: 0/100
Progress: 10/100
Progress: 20/100
Progress: 30/100
Progress: 40/100
Progress: 50/100
Progress: 60/100
Progress: 70/100
Progress: 80/100
Progress: 90/100

Metric                   GPT-4o      Haiku     Gemini
Precision                 0.908      0.888      0.925
Recall                    0.934      0.922      0.927
F1 Score                  0.915      0.900      0.925
Avg Latency (s)           0.948      1.065      0.741
Saved to ../logs/conll_baseline_20260526_132456.csv
